In [1]:
from utils import get_gsm8k_df

df = get_gsm8k_df(method="ReWOO", include=["model_history"])
df.head()

gemma-3-27b-it.xlsx
qwen3_1.7b.xlsx
smollm2_360m.xlsx
qwen2-math_1.5b.xlsx
qwen3_0.6b.xlsx
qwen2.5_0.5b.xlsx
gemma3_1b.xlsx
deepseek-r1_1.5b.xlsx
llama3.2_1b.xlsx


,Dataset,Method,Model,question,target_answer,response,is_correct,input_tokens,output_tokens,total_tokens,reasoning,Error Class,model_history,Error Type
0,GSM8K,ReWOO,google/gemma-3-27b-it,Janet’s ducks lay 16 eggs per day. She eats th...,Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eg...,18.0000000000000,True,865.0,0.0,865.0,Plan: Calculate the total number of eggs Janet...,NaN,[{'plan': {'steps': [['Calculate the total num...,NaN
1,GSM8K,ReWOO,google/gemma-3-27b-it,A robe takes 2 bolts of blue fiber and half th...,It takes 2/2=<<2/2=1>>1 bolt of white fiber\nS...,3.00000000000000,True,711.0,0.0,711.0,Plan: Calculate the amount of white fiber need...,NaN,[{'plan': {'steps': [['Calculate the amount of...,NaN
2,GSM8K,ReWOO,google/gemma-3-27b-it,Josh decides to try flipping a house. He buys...,The cost of the house and repairs came out to ...,195000.000000000,False,938.0,0.0,938.0,Plan: Calculate the total cost of the house in...,Wrong Reasoning,[{'plan': {'steps': [['Calculate the total cos...,Execution Error
3,GSM8K,ReWOO,google/gemma-3-27b-it,James decides to run 3 sprints 3 times a week....,He sprints 3*3=<<3*3=9>>9 times\nSo he runs 9*...,540,True,742.0,0.0,742.0,Plan: Calculate the total distance James runs ...,NaN,[{'plan': {'steps': [['Calculate the total dis...,NaN
4,GSM8K,ReWOO,google/gemma-3-27b-it,"Every day, Wendi feeds each of her chickens th...","If each chicken eats 3 cups of feed per day, t...",20,True,966.0,0.0,966.0,Plan: Calculate the total amount of feed Wendi...,NaN,[{'plan': {'steps': [['Calculate the total amo...,NaN


In [2]:
df["Error Class"].value_counts(normalize=True)

Error Class
Wrong Reasoning        0.527907
Plan Format Error      0.327907
Tool Error             0.095930
Plan-Solve Mismatch    0.019767
Overthinking           0.013953
Loop                   0.011628
Tool + Format Error    0.002907
Name: proportion, dtype: float64

In [3]:
import re
from typing import List, Tuple


def parse_plan(plan: str) -> List[Tuple[str, str, str, str]]:
    # remove <think> tags
    plan = re.sub(r'<think>.*?</think>', '', plan, flags=re.DOTALL)

    results = []

    # regex for normal Plan lines
    regex_pattern = r"Plan:\s*(.+?)\s*(#E\d+)\s*=\s*(\w+)\[([^\]]+)\]"
    matches = re.findall(regex_pattern, plan)
    results.extend(matches)

    # Find lines like: #E3 = Calculator[#E2 + 46]
    # If not already in results, add them with a default plan
    calc_lines = re.findall(r"(#E\d+)\s*=\s*(\w+)\[([^\]]+)\]", plan)
    known_ids = {eid for _, eid, _, _ in results}
    for eid, func, args in calc_lines:
        if eid not in known_ids:
            # Provide a placeholder or autogenerated plan
            auto_plan = f"(Auto) Compute {eid} using {func}[{args}]"
            results.append((auto_plan, eid, func, args))

    # put to string together
    str_result = ""
    for i in range(len(results)):
        str_result += f"Plan: {results[i][0]} {results[i][1]} = {results[i][2]}[{results[i][3]}]\n"
    if "Plan: Plan: " in str_result:
        str_result = str_result.replace("Plan: Plan: ", "Plan: ")
    return str_result

def parse_plan(plan: str) -> str:
    # remove <think> tags
    plan = re.sub(r'<think>.*?</think>', '', plan, flags=re.DOTALL)
    plan = re.sub(",", "", plan, flags=re.DOTALL)  # remove commas
    results: List[Tuple[str, str, str, str]] = []
    defined_ids = set()

    # regex for normal Plan lines
    regex_pattern = r"Plan:\s*(.+?)\s*(#E\d+)\s*=\s*(\w+)\[([^\]]+)\]"
    matches = re.findall(regex_pattern, plan)

    for desc, eid, func, args in matches:
        if eid not in defined_ids:
            results.append((desc.strip(), eid, func, args))
            defined_ids.add(eid)

    # fallback for lines without a "Plan:"
    fallback_pattern = r"(#E\d+)\s*=\s*(\w+)\[([^\]]+)\]"
    fallback_matches = re.findall(fallback_pattern, plan)

    for eid, func, args in fallback_matches:
        if eid not in defined_ids:
            auto_plan = f"(Auto) Compute {eid} using {func}[{args}]"
            results.append((auto_plan, eid, func, args))
            defined_ids.add(eid)

    # Format as a string if needed
    output = ""
    
    for desc, eid, func, args in results:
        output += f"Plan: {desc} {eid} = {func}[{args}]\n"

    if "Plan: Plan: " in output:
        output = output.replace("Plan: Plan: ", "Plan: ")
    if len(output) < 50:
        print(f"Parsed plan: {output.strip()}")
        print(f"Original plan: {plan.strip()}")
    return output.strip()

df["Adjusted Plan"] = df["reasoning"].apply(parse_plan)
df["Adjusted Plan"].iloc[0]

Parsed plan: 
Original plan: <think>
Okay let's tackle this problem step by step. So John is driving for 3 hours at 60 mph then turns around. He tries to get home in 4 hours but has some delays. Let me break down the details.

First the initial part: he drives for 3 hours at 60 mph. To find out how far he went during that time I need to calculate 3 hours multiplied by 60 mph. That would be 3*60 which is 180 miles. So #E1 would be 180.

Then he turns around. But he has a total of 4 hours to get home. However he spends the first 2 hours in standstill traffic. Wait so he's not driving those 2 hours. So the remaining time after the initial 3 hours is 4 - 3 = 1 hour? Wait no. Wait the total time is 4 hours. But he drove 3 hours initially then had 1 hour left. But during that 1 hour he had some delays. Let me check the problem again.

The problem says he tries to get home in 4 hours but spends the first 2 hours in standstill traffic. Then the next half-hour driving at 30 mph before being abl

'Plan: Calculate the total number of eggs Janet uses for breakfast and baking by adding the number of eggs she eats and the number she uses for baking. #E1 = Calculator[3 + 4]\nPlan: Determine the number of eggs Janet has left to sell by subtracting the total number of eggs used from the total number of eggs laid. #E2 = Calculator[16 - #E1]\nPlan: Calculate Janet’s daily earnings by multiplying the number of eggs she sells by the price per egg. #E3 = Calculator[#E2 * 2]'

In [4]:
from math_datasets.generators.rewoo_v1 import PlanExecutor

plan_executor = PlanExecutor()

def execute_plan(adjusted_plan):
    try:
        s = plan_executor.follow_plan(adjusted_plan)
        return s[-1]["solve"]["result"]
    except Exception as e:
        print(f"Error executing plan: {e}")
        print(f"Plan: {adjusted_plan}")
        return None

df["AutoSolve"] = df["Adjusted Plan"].apply(execute_plan)

/opt/miniconda3/envs/MA312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MPS (Metal Performance Shaders) is available! Using Apple Silicon GPU.
Error executing plan: list index out of range
Plan: 
Error executing plan: list index out of range
Plan: 
Error executing plan: list index out of range
Plan: 
Error executing plan: list index out of range
Plan: 
Error executing plan: list index out of range
Plan: 
Error executing plan: list index out of range
Plan: 
Error executing plan: list index out of range
Plan: 
Error executing plan: list index out of range
Plan: 
Error executing plan: list index out of range
Plan: 
Error executing plan: list index out of range
Plan: 
Error executing plan: list index out of range
Plan: 
Error executing plan: list index out of range
Plan: 
Error executing plan: list index out of range
Plan: 
Error executing plan: list index out of range
Plan: 
Error executing plan: list index out of range
Plan: 
Error executing plan: list index out of range
Plan: 
Error executing plan: list index out of range
Plan: 
Error executing plan: list i

In [5]:
print(df[df["response"] == "Error occured."].iloc[1]["Adjusted Plan"])
print("-----")
print(df[df["response"] == "Error occured."].iloc[1]["AutoSolve"])

Plan: Calculate regular pay for 40 hours. #E1 = Calculator[40 * 10]
Plan: Calculate overtime pay for 5 hours. #E2 = Calculator[5 * (1.2 * 10)]
Plan: Add regular and overtime pay. #E3 = Calculator[#E1 + #E2]
-----
460.0


In [6]:
df[df["AutoSolve"].isnull()].iloc[0]["AutoSolve"]

In [7]:
# count how many of the AutoSolve are correct
from math_datasets.datasets import Dataset

def extract_answer(answer):
    if answer is None:
        return None
    return Dataset.extract_answer(answer)

df["target_answer_float"] = df["target_answer"].apply(extract_answer)
df["AutoSolve_float"] = df["AutoSolve"].apply(extract_answer)

df["AutoSolve_correct"] = df["AutoSolve_float"] == df["target_answer_float"]
df["AutoSolve_correct"].value_counts(), df["is_correct"].value_counts()

(AutoSolve_correct
 False    1970
 True      730
 Name: count, dtype: int64,
 is_correct
 False    1721
 True      979
 Name: count, dtype: int64)

In [8]:
df.groupby(["Model"]).agg({
    "AutoSolve_correct": "mean",
    "is_correct": "mean",
})

,AutoSolve_correct,is_correct
Model,,
google/gemma-3-27b-it,0.873333,0.890000
ollama/deepseek-r1_1.5b,0.506667,0.773333
ollama/gemma3_1b,0.043333,0.113333
ollama/llama3.2_1b,0.046667,0.040000
ollama/qwen2-math_1.5b,0.013333,0.003333
ollama/qwen2.5_0.5b,0.126667,0.123333
ollama/qwen3_0.6b,0.346667,0.553333
ollama/qwen3_1.7b,0.456667,0.743333
ollama/smollm2_360m,0.020000,0.023333
